[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/patterns/langgraph_patterns.ipynb)

# Seven matched patterns with LangGraph

This notebook uses the same household-food-waste task, fixtures, schemas, tools and checks as the plain-Python baseline. LangGraph changes only orchestration. Mock runtime is under one minute on a CPU; local inference is practical but slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import asyncio
import json
from pathlib import Path
from typing import ClassVar

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import (  # noqa: E402
    CriticDecision,
    Message,
    MessageRole,
    PlanDecision,
    RouteDecision,
    ToolDefinition,
)

fixture_data = json.loads(
    (ROOT / "data/research_assistant/pattern_mock_scenarios.json").read_text()
)
catalogue = fixture_data["catalogue"]
task = "Which interventions reduce household food waste, and under what conditions?"

## Shared decision boundary
The model proposes structured state; graph nodes validate and execute it. Catalogue text is untrusted data.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Claim(StrictModel):
    schema_id: ClassVar[str] = "tutorial.claim.v1"
    claim: str
    source_id: str
    supported: bool


class Answer(StrictModel):
    schema_id: ClassVar[str] = "tutorial.pattern_answer.v1"
    answer: str
    source_ids: tuple[str, ...]


class ParallelDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.parallel.v1"
    queries: tuple[str, ...] = Field(min_length=2, max_length=3)
    aggregation: str = Field(pattern="^validated_union$")


class OrchestrationDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.orchestration.v1"
    assignments: tuple[dict, ...] = Field(min_length=1, max_length=2)


def model_for(name):
    return DeterministicMockClient(
        ScriptedScenarioFixture.model_validate(
            {
                "fixture_version": fixture_data["fixture_version"],
                "scenario": name,
                "provenance": fixture_data["provenance"],
                "responses": fixture_data["scenarios"][name],
            }
        )
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


def search(query, limit=3):
    terms = set(query.casefold().split())
    return [r for r in catalogue if terms & set(" ".join(r["topics"]).casefold().split())][:limit]


search_tool = ToolDefinition(
    name="search_catalogue",
    description="Search the bounded catalogue.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string"}, "max_results": {"type": "integer"}},
        "required": ["query", "max_results"],
    },
)


async def propose(client, schema, instruction):
    response = await client.generate([user(instruction)], response_schema=schema)
    return schema.model_validate(response.structured_output)

## 1–2. Prompt chaining and routing
Edges make sequencing and route selection explicit. Unknown routes end safely; a valid but semantically wrong route remains a limitation.

In [ ]:
chain_model = model_for("prompt_chaining")


async def extract_node(state):
    claim = await propose(
        chain_model, Claim, "Extract one supported claim with source_id food-waste-001."
    )
    return {"claim": claim.model_dump(), "trace": state["trace"] + ["extract"]}


async def synthesise_node(state):
    answer = await propose(chain_model, Answer, f"S synthesise only from {state['claim']}")
    return {
        "answer": answer.model_dump(),
        "trace": state["trace"] + ["synthesise"],
        "stop": "criteria_met",
    }


chain = StateGraph(dict)
chain.add_node("extract", extract_node)
chain.add_node("synthesise", synthesise_node)
chain.add_edge(START, "extract")
chain.add_edge("extract", "synthesise")
chain.add_edge("synthesise", END)
chain_state = await chain.compile().ainvoke({"trace": []})
route_model = model_for("routing")


async def classify(state):
    route = await propose(route_model, RouteDecision, f"Route: {task}")
    return {"route": route.route, "trace": ["route_validated"]}


def research(state):
    return {
        "results": search("meal planning portion"),
        "trace": state["trace"] + ["research"],
        "stop": "criteria_met",
    }


def clarify(state):
    return {"results": [], "trace": state["trace"] + ["clarify"], "stop": "safe_fallback"}


routing = StateGraph(dict)
routing.add_node("classify", classify)
routing.add_node("research", research)
routing.add_node("clarify", clarify)
routing.add_edge(START, "classify")
routing.add_conditional_edges(
    "classify", lambda s: s["route"] if s["route"] in {"research", "clarify"} else "clarify"
)
routing.add_edge("research", END)
routing.add_edge("clarify", END)
route_state = await routing.compile().ainvoke({})

## 3–5. Parallelisation, ReAct and planner–executor
Graph nodes expose concurrent branches, tool boundaries and plan execution. Budgets stop ReAct after two transitions and the executor rejects unsatisfied dependencies.

In [ ]:
parallel_model = model_for("parallelisation")


async def parallel_node(state):
    decision = await propose(
        parallel_model, ParallelDecision, "Propose two independent evidence searches."
    )
    batches = await asyncio.gather(*(asyncio.to_thread(search, q, 2) for q in decision.queries))
    records = {r["source_id"]: r for batch in batches for r in batch}
    return {
        "decision": decision.model_dump(),
        "source_ids": sorted(records),
        "trace": ["fan_out", "validated_union"],
        "stop": "criteria_met",
    }


parallel = StateGraph(dict)
parallel.add_node("fan_out_and_join", parallel_node)
parallel.add_edge(START, "fan_out_and_join")
parallel.add_edge("fan_out_and_join", END)
parallel_state = await parallel.compile().ainvoke({})
react_model = model_for("react")


async def react_decide(state):
    response = await react_model.generate(
        [user("Choose the next bounded tool call.")], tools=[search_tool]
    )
    call = response.tool_calls[0]
    if call.name != "search_catalogue":
        return {"stop": "invalid_tool", "trace": ["rejected"]}
    return {"call": call.model_dump(), "trace": ["tool_validated"], "steps": 1}


def react_tool(state):
    records = search(state["call"]["arguments"]["query"], state["call"]["arguments"]["max_results"])
    return {
        "records": records,
        "steps": state["steps"],
        "trace": state["trace"] + ["tool_executed"],
        "stop": "criteria_met",
    }


react = StateGraph(dict)
react.add_node("decide", react_decide)
react.add_node("tool", react_tool)
react.add_edge(START, "decide")
react.add_conditional_edges(
    "decide", lambda s: END if s.get("stop") else "tool", {"tool": "tool", END: END}
)
react.add_edge("tool", END)
react_state = await react.compile().ainvoke({})
plan_model = model_for("planner_executor")


async def plan_node(state):
    return {
        "plan": (
            await propose(plan_model, PlanDecision, "Create a bounded search then synthesis plan.")
        ).model_dump(),
        "trace": ["plan_validated"],
    }


def execute_plan(state):
    steps = state["plan"]["steps"]
    valid = len(steps) == 3 and all(
        steps[index]["depends_on"] == (steps[index - 1]["step_id"],) for index in (1, 2)
    )
    return {
        "trace": state["trace"] + (["executed"] if valid else ["dependency_stop"]),
        "stop": "criteria_met" if valid else "invalid_plan",
    }


planner = StateGraph(dict)
planner.add_node("plan", plan_node)
planner.add_node("execute", execute_plan)
planner.add_edge(START, "plan")
planner.add_edge("plan", "execute")
planner.add_edge("execute", END)
plan_state = await planner.compile().ainvoke({})

## 6–7. Critic–reviser and orchestrator–worker
Conditional edges bound revision; the orchestrator validates assignments before concurrent workers. The mock does not measure real model disagreement.

In [ ]:
critic_model = model_for("critic_reviser")


async def draft_node(state):
    draft = await propose(critic_model, Answer, "Draft an answer.")
    return {"draft": draft.model_dump(), "revisions": 0, "trace": ["draft"]}


async def critique(state):
    decision = await propose(critic_model, CriticDecision, f"Check citations in {state['draft']}")
    return {
        "draft": state["draft"],
        "revisions": state["revisions"],
        "decision": decision.model_dump(),
        "trace": state["trace"] + [f"critic:{'accept' if decision.accepted else 'revise'}"],
    }


async def revise(state):
    draft = await propose(critic_model, Answer, f"Revise once: {state['decision']['issues']}")
    return {
        "draft": draft.model_dump(),
        "revisions": state["revisions"] + 1,
        "trace": state["trace"] + ["revised"],
        "stop": "criteria_met",
    }


def critic_route(state):
    return "revise" if not state["decision"]["accepted"] and state["revisions"] < 1 else END


critic = StateGraph(dict)
critic.add_node("draft", draft_node)
critic.add_node("critic", critique)
critic.add_node("revise", revise)
critic.add_edge(START, "draft")
critic.add_edge("draft", "critic")
critic.add_conditional_edges("critic", critic_route, {"revise": "revise", END: END})
critic.add_edge("revise", END)
critic_state = await critic.compile().ainvoke({"trace": []}, {"recursion_limit": 6})
orchestrator_model = model_for("orchestrator_worker")


async def assign(state):
    return {
        "assignment": (
            await propose(
                orchestrator_model, OrchestrationDecision, "Assign two bounded reviewers."
            )
        ).model_dump(),
        "trace": ["assignments_validated"],
    }


async def workers(state):
    allowed = {"intervention_reviewer", "planning_reviewer"}
    assignments = state["assignment"]["assignments"]
    if any(a["worker"] not in allowed for a in assignments):
        return {"stop": "invalid_assignment"}
    batches = await asyncio.gather(*(asyncio.to_thread(search, a["query"], 2) for a in assignments))
    answer = await propose(orchestrator_model, Answer, f"S synthesise worker evidence {batches}")
    return {
        "answer": answer.model_dump(),
        "worker_count": len(batches),
        "trace": state["trace"] + ["workers_completed", "synthesis_validated"],
        "stop": "criteria_met",
    }


orchestrator = StateGraph(dict)
orchestrator.add_node("assign", assign)
orchestrator.add_node("workers", workers)
orchestrator.add_edge(START, "assign")
orchestrator.add_edge("assign", "workers")
orchestrator.add_edge("workers", END)
orchestrator_state = await orchestrator.compile().ainvoke({})

## Evaluation summary
Each compiled graph terminated explicitly and retained a trace. Failure demonstrations: unknown routes fall back, invalid tools are rejected, invalid dependencies stop, and revision is capped at one.

In [ ]:
pattern_evaluations = {
    "prompt_chaining": chain_state["stop"] == "criteria_met" and len(chain_state["trace"]) == 2,
    "routing": route_state["stop"] == "criteria_met" and len(route_state["results"]) == 2,
    "parallelisation": parallel_state["source_ids"] == ["food-waste-001", "food-waste-002"],
    "react": react_state["stop"] == "criteria_met" and react_state["steps"] <= 2,
    "planner_executor": plan_state["stop"] == "criteria_met",
    "critic_reviser": critic_state["revisions"] == 1 and critic_state["stop"] == "criteria_met",
    "orchestrator_worker": orchestrator_state["stop"] == "criteria_met"
    and orchestrator_state["worker_count"] == 2,
}
pattern_limitations = {
    "prompt_chaining": "early errors propagate",
    "routing": "valid routes can be semantically wrong",
    "parallelisation": "branches can duplicate work",
    "react": "tool loops require strict budgets",
    "planner_executor": "invalid dependencies stop execution",
    "critic_reviser": "one revision may be insufficient",
    "orchestrator_worker": "worker findings can conflict",
}
assert set(pattern_limitations) == set(pattern_evaluations)
assert all(pattern_evaluations.values()), pattern_evaluations
{
    "framework": "LangGraph",
    "evaluation": pattern_evaluations,
    "limitations": pattern_limitations,
    "limitation": "Graph topology does not ensure semantic correctness.",
}